# XGBOOST — Predikcia priemyselnej produkcie SR

In [6]:
# import potrebnych kniznic a konfiguracia notebooku

import importlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from xgboost import XGBRegressor
import shap

import functions
importlib.reload(functions)
from functions import forward_selection
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (13, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("Importy uspesne.")


Importy uspesne.


In [3]:
# konfiguracia a stiahnutie dat
# vypisanie info o datach a kontrola chybajucich hodnot

DATA_PATH  = r"C:\Users\adamv\Desktop\ADAM\VS\Diplomovka\excel\data.xlsx"
SHEET_NAME = "all"

df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, engine="openpyxl")

df_raw["datum"] = pd.to_datetime(df_raw["datum"])
df_raw = df_raw.set_index("datum").sort_index()
df_raw = df_raw.asfreq("MS")

print(f"Nacitane: {df_raw.shape[0]} riadkov, {df_raw.shape[1]} stlpcov")
print(f"Obdobie : {df_raw.index.min().date()} az {df_raw.index.max().date()}")
print("Stlpce  :", list(df_raw.columns))

missing = df_raw.isnull().sum()
if missing.any():
    print("\nChybajuce hodnoty:")
    print(missing[missing > 0])
else:
    print("\nZiadne chybajuce hodnoty.")


target = "pp_sa_new"
exog_cols = ["export_celkovy_sa", "trzby_priemysel_sa", "cpi", "crisis_dummy"]

df = df_raw.copy()

df["crisis_dummy"] = 0
df.loc["2008-01-01", "crisis_dummy"] = 1


Nacitane: 312 riadkov, 13 stlpcov
Obdobie : 2000-01-01 az 2025-12-01
Stlpce  : ['pp', 'cpi', 'ipcv_priemysel', 'export_celkovy_sa', 'trzby_priemysel_sa', 'kurz_usd_eur', 'sadzba_3m_euribor', 'uvery_celkom', 'vklady_nefinancne_podniky', 'prod_dev', 'prod_exp', 'ind_conf', 'fin_stock']

Ziadne chybajuce hodnoty.


In [ ]:
# kontrola stacionarity premennych
# premenna je stacionarna len ak ADF zamieta H0 A KPSS nezamieta H0.

EXOG_CANDIDATES = df.columns.tolist()

df_to_test = df[EXOG_CANDIDATES].copy()

print("=== ADF test — urovne ===")
non_stationary = functions.adf_test(df)

print("\n=== KPSS test — urovne ===")
for col in EXOG_CANDIDATES:
    stat, pval, _, _ = kpss(df[col].dropna(), regression="c", nlags="auto")
    verdict = "nestacionarna" if pval < 0.05 else "stacionarna"
    print(f"  {col}: stat={stat:.4f}, p={pval:.4f}  [{verdict}]")

print("\nNestacionarne (podla ADF):", non_stationary)


In [ ]:
# diferencovanie nestacionarnych premennych

df_diff = df[non_stationary].diff(1).dropna()
print("=== ADF test — prve diferencie ===")
non_stationary_diff = functions.adf_test(df_diff)

if non_stationary_diff:
    print("\nPremenné stale nestacionarne po 1. diferencii (zvazit I(2)):")
    print(non_stationary_diff)
else:
    print("\nVsetky premenne su stacionarne po 1. diferencii.")
